# 03 decode 时训练：ESamp

**场景**：有一类算法要在**生成的同时学习**：每个请求带着自己的一小份参数，每生成一个 token
就用这个 token 上的信号更新一次，更新后的参数再影响后面的生成。测试时训练（test-time training）
的很多变体都是这个形状。这篇用 ESamp 做例子：

> 每个请求训练一个小网络（distiller），用第 0 层的 hidden 预测最后一层的 hidden。采样时，把这个
> 浅层预测对应的 logits 按强度 beta 从模型的 logits 里对减：浅层就能猜到的 token 被压低，深层额外
> 贡献的 token 被抬高。

在推理引擎里手写这件事很费劲：autograd 要在推理模式里打开；每个请求要有自己的参数和 AdamW 状态，
请求来来去去，显存要分配和回收；反向和优化器步要和下一步 forward 重叠，否则吞吐直接腰斩；最后
还想让它们都进 CUDA graph。

这篇你只写：小网络本身（普通 `nn.Module`）、一个 loss、一个预测。然后在 128 个并发请求上测两件事：
采样多样性有没有变化，吞吐还剩多少。

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # 只显示 vLLM 的报错，省掉启动日志

import gc
import time
from dataclasses import dataclass

import torch
from torch import nn
from torch.nn import functional as F
from vllm import SamplingParams

import pllm

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# 128 个并发请求（8 个 prompt × 每个 16 个样本）。
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=128, max_num_batched_tokens=8192,
                          gpu_memory_utilization=0.5, seed=7)

## 第一步：小网络就是普通的 PyTorch

一个低秩残差块加 layer norm。`up` 初始化为零，所以训练开始前它只是把输入做一次 layer norm。

In [2]:
class Distiller(nn.Module):
    def __init__(self, width: int, rank: int) -> None:
        super().__init__()
        self.down = nn.Linear(width, rank, bias=False)
        self.up = nn.Linear(rank, width, bias=False)
        nn.init.zeros_(self.up.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = x + self.up(F.silu(self.down(x)))
        return F.layer_norm(y, y.shape[-1:])

## 第二步：负载

- `distiller` 是一份 `pllm.RequestState`：**每个请求一份**。`module=` 给出怎么构造这个网络
  （`m` 是加载模型后才知道的维度，`m.hidden_size` 对 0.5B 模型是 896），它的参数就是状态，
  `optimizer=` 说明用什么更新它。请求结束时这份状态连同 AdamW 的动量一起回收。
- `loss` 带 `@pllm.train(state="distiller")`：返回**每一行**的 loss（一行是一个请求当前的 token）。
  框架负责屏蔽填充行、求平均（`reduction="state_mean"` 表示先在每个请求内平均）、反向、应用 AdamW、
  发布新版本。`pllm.module("distiller")` 拿到的是"每一行用自己那个请求的参数"的网络。
- `predict` 带 `@pllm.guide_sampling(beta=...)`：返回的预测会被送进 LM head，采样分布变成
  `(1 + beta) * logits - beta * 预测的 logits`。`rows="trained"` 让还没训练过的请求不受影响；
  `trained` 方法用 `pllm.version` 读每个请求的状态被更新过几次。

In [3]:
@dataclass(frozen=True)
class MiniESamp(pllm.Workload):
    beta: float
    rank: int = 4

    @property
    def distiller(self) -> pllm.RequestState:
        return pllm.RequestState(module=lambda m: Distiller(m.hidden_size, self.rank), optimizer=pllm.AdamW(1e-3))

    def trained(self, versions: torch.Tensor = pllm.version("distiller")) -> torch.Tensor:
        return versions > 0

    @pllm.train(state="distiller", reduction="state_mean")
    def loss(self, shallow: torch.Tensor = pllm.hidden(0), deep: torch.Tensor = pllm.hidden(-1),
             f: Distiller = pllm.module("distiller")) -> torch.Tensor:
        return (f(shallow) - deep).square().mean(-1)

    @pllm.guide_sampling(beta=pllm.Attr("beta"), rows="trained")
    def predict(self, shallow: torch.Tensor = pllm.hidden(0), f: Distiller = pllm.module("distiller")) -> torch.Tensor:
        return f(shallow)

负载写成 frozen dataclass，`beta` 和 `rank` 是它的字段；`pllm.Attr("beta")` 让装饰器的参数读这个字段，
这样同一个类可以用不同的 `beta` 实例化。

## 第三步：128 个请求，看吞吐和输出

8 个 prompt，每个采样 16 个续写，各 256 个 token。三组对照：

- **原生 vLLM**：不带负载；
- **ESamp，beta = 0**：每个请求照常训练小网络，但不引导采样。训练不应该改变任何一个采样结果；
- **ESamp，beta = 0.5**：训练并引导采样。

每组先跑一轮预热（含 CUDA graph 捕获），再计时两轮取较快的一轮。三组用同一个随机种子，所以可以逐个
样本比较输出。

In [4]:
prompts = [f"Explain the mathematical idea numbered {i}:" for i in range(8)]
sampling = SamplingParams(n=16, max_tokens=256, temperature=0.8, top_p=0.95, ignore_eos=True, seed=7)


def measure(llm) -> tuple[float, list[str]]:
    llm.generate(prompts, sampling, use_tqdm=False)  # 预热
    best = 0.0
    for _ in range(2):
        torch.cuda.synchronize()
        start = time.perf_counter()
        outputs = llm.generate(prompts, sampling, use_tqdm=False)
        torch.cuda.synchronize()
        tokens = sum(len(sample.token_ids) for output in outputs for sample in output.outputs)
        best = max(best, tokens / (time.perf_counter() - start))
    return best, [sample.text for output in outputs for sample in output.outputs]


plain = pllm.make_llm(CONFIG)
plain_rate, plain_texts = measure(plain)
del plain
gc.collect()
torch.cuda.empty_cache()
print(f"原生 vLLM        : {plain_rate:6.0f} token/s")

for beta in (0.0, 0.5):
    with pllm.make_llm(CONFIG, workload=MiniESamp(beta)) as llm:
        rate, texts = measure(llm)
        report = llm.diagnostics()
    same = sum(a == b for a, b in zip(texts, plain_texts))
    print(f"ESamp beta={beta:<4}: {rate:6.0f} token/s，为原生的 {rate / plain_rate:.0%}；"
          f"{same}/{len(texts)} 个样本与原生相同；累计训练 {report.trained_rows} 行；"
          f"eager 原因 {report.eager_reason!r}")

原生 vLLM        :  40243 token/s
ESamp beta=0.0 :  37848 token/s，为原生的 94%；127/128 个样本与原生相同；累计训练 97920 行；eager 原因 ''
ESamp beta=0.5 :  30314 token/s，为原生的 75%；0/128 个样本与原生相同；累计训练 97920 行；eager 原因 ''


三行结果说明三件事：

- **训练不打扰生成**：beta = 0 时每个请求都在训练（"累计训练"是三轮生成里所有请求、所有 token 上的训练行数），
  128 个样本里有 127 个和原生 vLLM 逐字相同。剩下一个在中途分叉，是浮点舍入层面的差异：带负载的引擎要从
  forward 里取出 hidden，编译出的 graph 和原生不完全相同，256 个 token 的随机采样偶尔会在几乎打平的地方走向另一边；
- **代价可以接受**：每个 token、每个请求都做一次前向、反向和 AdamW，吞吐仍是原生的 94%；
- **引导确实生效**：beta = 0.5 时所有样本都变了，吞吐降到原生的 75%，多出来的是每步对预测再做一次 LM head
  和对采样分布的修改。

这些数字来自本地 RTX 4090 上的一次运行，不同机器、不同轮次会有几个百分点的浮动。

## 框架替你做了什么

- **每个请求的参数和优化器状态**：128 份小网络加各自的 AdamW 动量，按页分配在启动时就预算好的显存里，
  请求结束即回收；
- **CUDA graph**：`eager 原因` 为空，说明模型 forward、你的 loss、反向和 AdamW 更新都在 CUDA graph 里；
- **重叠**：训练放在旁路 stream 上，和下一步 forward 同时跑。代价是一步的延迟：第 N 步的更新在第 N+2 步
  才被预测读到。需要严格顺序时用 `pllm.ExecutionOptions(publish_lag=0)`；
- **调度**：你没有说训练在哪个 stream、什么时候同步。pllm 的规划器从声明里看出 `loss` 只写
  `distiller`、`predict` 只读它，据此安排顺序和重叠。

完整的 ESamp 在 `pllm.workloads.ESampWorkload`，和这里的 `MiniESamp` 是同一个结构，只是小网络多了门控。

下一篇 [04 改写 forward](04_custom_forward.ipynb) 换一个方向：不加旁路计算，而是改变模型本身跑哪些层。